In [4]:
file = '../Data/scfa_predictions/data/raw_data/'

In [6]:
import pandas as pd
df = pd.read_csv(file)
df

IsADirectoryError: [Errno 21] Is a directory: '../Data/scfa_predictions/data/raw_data/'

In [9]:
import os
import requests
import urllib.parse
import time

# ===== 输入 =====
tax_line = "Bacteria,Proteobacteria,Gammaproteobacteria,Aeromonadales,Aeromonadaceae,Oceanimonas,Oceanimonas_sp._GK1"
species_name = tax_line.split(",")[-1].replace("_", " ")
encoded_name = urllib.parse.quote(species_name)

# out_dir = os.path.abspath("genomes_faa")
out_dir = os.path.abspath("./StudyA_genome_gz/")
os.makedirs(out_dir, exist_ok=True)

print(f"\n=== Searching NCBI for: {species_name} ===")

# ===== 第一步：用 NCBI taxonomy 搜索 TaxID =====
tax_url = f"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/taxonomy/name/{encoded_name}"
r = requests.get(tax_url)

if r.status_code != 200:
    print(f"❌ Taxonomy query failed ({r.status_code})")
    print(r.text)
else:
    try:
        data = r.json()
        if "taxonomy_nodes" not in data or not data["taxonomy_nodes"]:
            print(f"❌ No taxonomy found for {species_name}")
        else:
            tax_id = data["taxonomy_nodes"][0]["taxonomy"]["tax_id"]
            print(f"✅ Found TaxID: {tax_id}")

            # ===== 第二步：用 TaxID 获取基因组 accession =====
            genome_url = f"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/genome/taxon/{tax_id}/dataset_report"
            r2 = requests.get(genome_url)
            if r2.status_code != 200:
                print(f"❌ Genome dataset query failed ({r2.status_code})")
            else:
                data2 = r2.json()
                reports = data2.get("reports", [])
                if not reports:
                    print(f"❌ No genome available for {species_name}")
                else:
                    accession = reports[0]["accession"]
                    print(f"✅ Found genome accession: {accession}")

                    # ===== 第三步：下载蛋白质序列 =====
                    faa_url = f"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/genome/accession/{accession}/download?include=protein.faa"
                    out_path = os.path.join(out_dir, f"{species_name.replace(' ', '_')}.zip")

                    print("⬇️ Downloading protein FASTA...")
                    with requests.get(faa_url, stream=True) as res:
                        if res.status_code == 200:
                            with open(out_path, "wb") as f:
                                for chunk in res.iter_content(chunk_size=8192):
                                    f.write(chunk)
                            print(f"✅ Saved to: {out_path}")
                        else:
                            print(f"❌ Download failed ({res.status_code})")
        time.sleep(1)
    except Exception as e:
        print("❌ Error parsing JSON response")
        print(str(e))



=== Searching NCBI for: Oceanimonas sp. GK1 ===


ConnectionError: HTTPSConnectionPool(host='api.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /datasets/v2alpha/taxonomy/name/Oceanimonas%20sp.%20GK1 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdfc94d6250>: Failed to establish a new connection: [Errno 101] Network is unreachable'))

In [1]:

out_dir = os.path.abspath("./StudyA_genome_gz/")



Searching and downloading protein FASTA for: Blautia sp. SC05B48


JSONDecodeError: Expecting value: line 1 column 1 (char 0)